# Copper reflectance in BBRsim — the Drude model

Every blackbody photon in a cryostat bounces off copper hundreds to tens of
thousands of times before it is absorbed. How many times is set by one number,
the absorptance $D = 1 - R$ of copper at millimetre wavelengths and cryogenic
temperature, and that number decides whether stray 4 K radiation reaches a
detector or dies in the walls. This notebook derives it, shows why the textbook
low-frequency formula cannot be used, and checks the model against measurement.

**Source of truth.** Every formula here is imported from
`tools/python/bbrsim/physics.py`, which mirrors the C++ in
`library/include/BBRMaterials.hh` and is unit-tested by `tools/python/tests/test_physics.py`.
Nothing is re-derived in this notebook, so it cannot drift from the simulation.
The written derivation is `docs/physics/copper_reflectance_model.md`.

**Running it.** Select the `bbrsim` conda environment as the kernel. That
environment has no Jupyter support yet, so install the kernel once with
`conda run -n bbrsim pip install ipykernel` (VS Code will also offer to do this
for you when you pick the kernel).
The kernel must also be able to import the `bbrsim` package. Install it once,
editable, with `conda run -n bbrsim pip install -e tools/python` from the
repository root, or start Jupyter from a shell that has sourced `bbrsim_env.sh`.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bbrsim import paths, physics as ph

DATA = paths.data_dir()   # $BBRSIMDATA, else the checkout's data/

# RRR values are ordered levels, so curves use one blue ramp, light to dark with
# increasing purity. Measurements are a different kind of thing, not another
# level, so they get a contrasting hue.
RRR_COLOR = {3: "#86b6ef", 6: "#3987e5", 100: "#1c5cab", 500: "#0d366b"}
MEASURED = "#eb6834"
INK, INK_2, INK_MUTED = "#0b0b0b", "#52514e", "#8a8a85"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": INK_MUTED, "axes.labelcolor": INK_2,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": INK_2, "ytick.color": INK_2, "text.color": INK,
    "grid.color": INK_MUTED, "grid.alpha": 0.18, "grid.linewidth": 0.6,
    "axes.grid": True, "axes.axisbelow": True, "figure.dpi": 110,
    "legend.frameon": False, "font.size": 10,
})

# Display copy of a table with the named columns formatted. Plain DataFrames
# render as tables in a notebook; pandas' .style accessor would need jinja2,
# which this environment does not carry.
def show(df, **fmt):
    out = df.copy()
    for col, spec in fmt.items():
        out[col] = out[col].map(lambda v: format(v, spec))
    return out


# The band BBRsim samples: the Planck CDF runs 10 GHz to 20 THz.
BAND = (10e9, 20e12)
print("data:", DATA)

## 1. Why the answer hinges on this number

A photon that is not absorbed is still in the cryostat. The mean number of
bounces a photon survives is $1/D$, so an absorptance of $5\times10^{-5}$ means
a 4 K photon wanders roughly twenty thousand reflections before it is
thermalised, and it can find a detector anywhere in that path. The table below
is the whole motivation: copper purity changes the survival length by more than
two orders of magnitude.

`planck_weighted_absorptance` averages $D(\nu)$ over the 4 K photon-number
spectrum across the band the simulation samples, so it is the number that
actually governs a run rather than a value at one convenient frequency.

In [ ]:
rows = []
for rrr in (3, 6, 100, 500):
    for T in (4.0, 20.0):
        D = ph.planck_weighted_absorptance(rrr, T, *BAND)
        rows.append({"RRR": rrr, "emitter T [K]": T,
                     "<D> Planck-weighted": D, "mean bounces 1/<D>": 1.0 / D})
summary = pd.DataFrame(rows)
show(summary, **{"<D> Planck-weighted": ".3e", "mean bounces 1/<D>": ",.0f",
                 "emitter T [K]": ".0f"})

## 2. One number in: the residual resistance ratio

Users never supply a conductivity. They supply **RRR**, the ratio of a sample's
room-temperature resistivity to its residual resistivity, which is what a
copper vendor quotes and what a four-wire measurement returns. Everything else
follows:

$$\sigma_{DC}(T) = \mathrm{RRR}\times\sigma_{RT},\qquad
  \tau = \frac{\sigma_{DC}\,m_e}{n_e e^2}$$

with $\sigma_{RT} = 5.96\times10^{7}$ S/m, a constant for all copper grades at
273 K that does not depend on purity. Below about 50 K the impurity term
dominates and the phonon contribution is negligible, which is why the same
expression serves the whole sub-Kelvin to liquid-nitrogen range.

The scattering time $\tau$ sets a corner frequency $1/(2\pi\tau)$. Below it the
electrons respond in phase with the field; above it they cannot keep up, and
the physics changes character. Note how far that corner moves with purity.

In [ ]:
rrr_values = [1, 3, 6, 100, 500]
tbl = pd.DataFrame({
    "RRR": rrr_values,
    "sigma_DC [S/m]": [ph.sigma_dc(r, 4.0) for r in rrr_values],
    "tau [ps]": [ph.drude_tau(r, 4.0) * 1e12 for r in rrr_values],
    "corner 1/(2 pi tau) [GHz]": [1.0 / (2 * np.pi * ph.drude_tau(r, 4.0)) / 1e9
                                  for r in rrr_values],
    "grade": ["disordered", "low-purity / cold-worked", "hydrogen-annealed",
              "standard OFHC", "ultra-pure crystal"],
})
show(tbl, **{"sigma_DC [S/m]": ".3e", "tau [ps]": ".3f",
             "corner 1/(2 pi tau) [GHz]": ",.1f"})

## 3. Hagen-Rubens cannot be used in this band

The Hagen-Rubens result, $D = \sqrt{8\varepsilon_0\omega/\sigma}$, is the
textbook absorptance of a good conductor, and it is what most cryogenic
estimates reach for. It is a **low-frequency limit**, valid only while
$\omega\tau \ll 1$. For standard OFHC copper at 4 K that corner sits near
64 GHz, below almost the entire band BBRsim samples, and Hagen-Rubens
over-predicts the loss above it, in the direction that makes a cryostat look
better sealed than it is.

The plot shows where each curve leaves the other. For low-purity copper the
corner is above 2 THz, so Hagen-Rubens is defensible across most of the band,
which is exactly why measurements on dirty copper at a few hundred GHz have
historically agreed with it.

In [ ]:
nu = np.logspace(np.log10(BAND[0]), np.log10(BAND[1]), 600)
fig, ax = plt.subplots(figsize=(8.2, 4.8))
for rrr in (3, 100):
    ax.loglog(nu / 1e9, ph.drude_absorptance(nu, rrr, 4.0),
              color=RRR_COLOR[rrr], lw=2,
              label=f"Drude, RRR = {rrr}")
    ax.loglog(nu / 1e9, ph.hagen_rubens_absorptance(nu, ph.sigma_dc(rrr, 4.0)),
              color=RRR_COLOR[rrr], lw=1.4, ls=":",
              label=f"Hagen-Rubens, RRR = {rrr}")
    f_break = 1.0 / (2 * np.pi * ph.drude_tau(rrr, 4.0)) / 1e9
    ax.axvline(f_break, color=RRR_COLOR[rrr], lw=0.9, ls="--")
    ax.annotate(f"corner, RRR {rrr}\n{f_break:,.0f} GHz", (f_break, 3e-6),
                rotation=90, fontsize=8, color=INK_2, ha="right", va="bottom")
ax.set_xlabel("frequency [GHz]")
ax.set_ylabel("absorptance $D = 1-R$")
ax.set_title("Hagen-Rubens over-predicts loss above the corner frequency",
             loc="left", color=INK)
ax.legend(fontsize=8, labelcolor=INK_2, loc="lower right")
plt.show()

## 4. The full complex Drude model, and the plateau it predicts

The model BBRsim uses keeps the conductivity complex:

$$\sigma(\omega) = \frac{\sigma_{DC}}{1 - i\omega\tau},\qquad
  \tilde\varepsilon = 1 + \frac{i\sigma(\omega)}{\varepsilon_0\omega},\qquad
  \tilde n = \sqrt{\tilde\varepsilon},\qquad
  R = \left|\frac{\tilde n - 1}{\tilde n + 1}\right|^2$$

Keeping the imaginary part is not optional. Substituting only
$\mathrm{Re}\,\sigma$ into Griffiths's real-conductivity forms drops the plasma
term in $\mathrm{Re}\,\tilde\varepsilon$ and overestimates absorptance by about
thirty times at 500 GHz. That was a real defect in this codebase, fixed in June
2026, and the self-test exists to keep it fixed.

Above the corner the absorptance stops falling and flattens onto the
**relaxation plateau**

$$D \simeq \frac{2}{\omega_p \tau},\qquad
  \omega_p = \sqrt{\frac{n_e e^2}{\varepsilon_0 m_e}} = 1.64\times10^{16}\ \mathrm{rad/s}$$

which is frequency-independent and depends on purity only. The plateau is the
practical regime for high-RRR copper across nearly the whole blackbody band, so
one number per material governs most of a simulation.

In [ ]:
EPS0 = 8.8541878128e-12
omega_p = np.sqrt(ph.N_E * ph.E_CHARGE**2 / (EPS0 * ph.M_E))

fig, ax = plt.subplots(figsize=(8.2, 5.0))
for rrr, color in RRR_COLOR.items():
    ax.loglog(nu / 1e9, ph.drude_absorptance(nu, rrr, 4.0), color=color, lw=2,
              label=f"RRR = {rrr}")
    plateau = 2.0 / (omega_p * ph.drude_tau(rrr, 4.0))
    ax.axhline(plateau, color=color, lw=0.9, ls="--")
    ax.annotate(rf"$2/(\omega_p\tau)$ = {plateau:.2e}", (BAND[1] / 1e9, plateau),
                textcoords="offset points", xytext=(-4, 4), ha="right",
                fontsize=8, color=color)

# The two cryogenic measurements the model is anchored to.
serov = pd.read_csv(os.path.join(DATA, "materials",
                                 "cu_serov_reference_points.csv"))
ax.scatter(serov["freq_GHz"], serov["D"], s=55, color=MEASURED, zorder=5,
           label="Serov 2016, measured at 4 K")
for _, r in serov.iterrows():
    ax.annotate(r["material"], (r["freq_GHz"], r["D"]),
                textcoords="offset points", xytext=(8, -3), fontsize=8,
                color=MEASURED)

# The photon-number peak of the 4 K spectrum, for orientation.
peak = ph.PLANCK_PEAK_U * ph.K_EV * 4.0 / ph.H_EVS / 1e9
ax.axvline(peak, color=INK_MUTED, lw=0.9)
ax.annotate(f"4 K photon-number peak\n{peak:.0f} GHz", (peak, 2e-6),
            textcoords="offset points", xytext=(6, 0), fontsize=8, color=INK_2)

ax.set_xlabel("frequency [GHz]")
ax.set_ylabel("absorptance $D = 1-R$")
ax.set_title("Full complex Drude: each purity flattens onto its own plateau",
             loc="left", color=INK)
ax.legend(fontsize=8, labelcolor=INK_2, loc="lower right")
plt.show()

### How close is the plateau approximation?

Worth checking rather than asserting. At 500 GHz the high-purity curves sit on
their plateau to a fraction of a percent, while RRR 3 is still well below its
corner and has not flattened at all, so the asymptote over-predicts it by about
sixty percent. That is the quantitative reason the two grades need different
treatment, and it is why the simulation tabulates the full model rather than
the plateau value.

In [ ]:
chk = pd.DataFrame({
    "RRR": list(RRR_COLOR),
    "D(500 GHz), full Drude": [ph.drude_absorptance(500e9, r, 4.0) for r in RRR_COLOR],
    "2/(omega_p tau)": [2.0 / (omega_p * ph.drude_tau(r, 4.0)) for r in RRR_COLOR],
})
chk["ratio"] = chk["2/(omega_p tau)"] / chk["D(500 GHz), full Drude"]
chk["on the plateau at 500 GHz?"] = np.where(
    abs(chk["ratio"] - 1) < 0.05, "yes", "no, still in the transition")
show(chk, **{"D(500 GHz), full Drude": ".3e", "2/(omega_p tau)": ".3e",
             "ratio": ".3f"})

## 5. Temperature

Below roughly 50 K the impurity term dominates and $\sigma \approx
\mathrm{RRR}\times\sigma_{RT}$ regardless of temperature, so a 4 K stage and a
20 K stage of the same copper behave almost identically. Above that,
Matthiessen's rule adds a phonon term and the curves separate. The simple
$1/T$ phonon scaling used here is a high-temperature limit; between about 10 and
50 K umklapp scattering breaks it, so treat the warm curves as indicative. That
regime matters for shield layers, never for the detector stage.

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.6))
temps = [4.0, 20.0, 77.0, 300.0]
shades = ["#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]
for T, color in zip(temps, shades):
    ax.loglog(nu / 1e9, ph.drude_absorptance(nu, 100, T), color=color, lw=2,
              label=f"{T:.0f} K")
ax.set_xlabel("frequency [GHz]")
ax.set_ylabel("absorptance $D = 1-R$")
ax.set_title("OFHC copper (RRR = 100): 4 K and 20 K are nearly the same curve",
             loc="left", color=INK)
ax.legend(fontsize=8, labelcolor=INK_2, title="stage temperature",
          loc="lower right")
plt.show()

## 6. What the simulation actually sees

Geant4 does not evaluate the model per photon. `BuildDrudeMaterial` tabulates
reflectivity at **24 log-spaced points from 10 GHz to 20 THz** and attaches them
to the material, and the boundary process interpolates that table. The grid
matches the Planck sampler's range exactly, so no photon is ever generated
outside it. The check below is that the discretisation costs nothing: on a
log-log scale the tabulated points lie on the continuous curve, because $D$ is
a smooth power law and then a constant.

In [ ]:
N_GRID = 24                       # BuildDrudeMaterial, library/include/BBRMaterials.hh
grid = np.exp(np.linspace(np.log(BAND[0]), np.log(BAND[1]), N_GRID))

fig, ax = plt.subplots(figsize=(8.2, 4.6))
for rrr in (3, 100):
    ax.loglog(nu / 1e9, ph.drude_absorptance(nu, rrr, 4.0),
              color=RRR_COLOR[rrr], lw=1.6, label=f"continuous model, RRR = {rrr}")
    ax.loglog(grid / 1e9, ph.drude_absorptance(grid, rrr, 4.0), "o",
              color=RRR_COLOR[rrr], markersize=5, markerfacecolor="none",
              label=f"24-point table, RRR = {rrr}")
ax.set_xlabel("frequency [GHz]")
ax.set_ylabel("absorptance $D = 1-R$")
ax.set_title(f"The {N_GRID}-point REFLECTIVITY table Geant4 interpolates",
             loc="left", color=INK)
ax.legend(fontsize=8, labelcolor=INK_2, loc="lower right")
plt.show()

worst = max(abs(ph.drude_absorptance(np.sqrt(grid[:-1] * grid[1:]), r, 4.0) /
                np.sqrt(ph.drude_absorptance(grid[:-1], r, 4.0) *
                        ph.drude_absorptance(grid[1:], r, 4.0)) - 1).max()
            for r in RRR_COLOR)
print(f"worst log-midpoint interpolation error across the grid: {worst:.2%}")

## 7. Validation against measurement

Two cryogenic data points anchor the model: Serov (2016) measured absorptance
on oxygen-free and hydrogen-annealed copper at 4 K. The aliases in the
simulation were assigned to reproduce them.

One of the two does not currently agree, and the discrepancy is recorded rather
than hidden. `HP_Cu` is mapped to RRR 6, a value that was originally derived
with Hagen-Rubens; under the full Drude model that mapping falls about thirteen
percent short of the measurement, because Hagen-Rubens over-predicts in that
range and so demanded too low an RRR. RRR 5 would land within a couple of
percent. `validation/check_cu_serov.py` fails on this row **by design** until the
alias is re-pinned, so the suite reports it as a known red rather than quietly
passing.

In [ ]:
val = serov.copy()
alias_rrr = {"OF_Cu": 3, "HP_Cu": 6}
val["RRR used"] = val["material"].map(alias_rrr)
val["D model"] = [ph.drude_absorptance(r["freq_GHz"] * 1e9, alias_rrr[r["material"]], 4.0)
                  for _, r in val.iterrows()]
val["model / measured"] = val["D model"] / val["D"]
val["RRR that would match"] = [
    min((abs(ph.drude_absorptance(r["freq_GHz"] * 1e9, k, 4.0) / r["D"] - 1), k)
        for k in range(1, 30))[1] for _, r in val.iterrows()]
val = val.rename(columns={"D": "D measured", "freq_GHz": "nu [GHz]"})[
    ["material", "nu [GHz]", "RRR used", "D measured", "D model",
     "model / measured", "RRR that would match"]]
show(val, **{"D measured": ".2e", "D model": ".2e", "model / measured": ".3f",
             "nu [GHz]": ".0f"})

### Room-temperature references

Two further tables ship with the repository and are worth plotting for context
rather than validation: Palik's handbook optical constants and the reflectivity
table shipped with Geant4's infrared examples. Both are room-temperature and
sit far above our band, so they constrain the high-frequency end of the model
and nothing else. The model is not fitted to them.

In [ ]:
mat_dir = os.path.join(DATA, "materials")
palik = pd.read_csv(os.path.join(mat_dir, "cu_palik_optical_constants.csv"))
g4ir = pd.read_csv(os.path.join(mat_dir, "cu_g4ir_reflectivity.csv"))

fig, ax = plt.subplots(figsize=(8.2, 4.6))
ax.loglog(nu / 1e9, ph.drude_absorptance(nu, 100, 4.0), color=RRR_COLOR[100],
          lw=2, label="Drude, RRR = 100, 4 K (BBRsim)")
ax.loglog(nu / 1e9, ph.drude_absorptance(nu, 100, 300.0), color=RRR_COLOR[3],
          lw=1.6, ls="--", label="Drude, RRR = 100, 300 K")
for df_, marker, label in ((palik, "s", "Palik handbook, 300 K"),
                           (g4ir, "^", "Geant4 IR table, 300 K")):
    m = (df_["freq_GHz"] > 1) & (df_["D"] > 0)
    ax.loglog(df_.loc[m, "freq_GHz"], df_.loc[m, "D"], marker, markersize=4,
              color=MEASURED if marker == "s" else INK_MUTED,
              markerfacecolor="none", linestyle="none", label=label)
ax.axvspan(BAND[0] / 1e9, BAND[1] / 1e9, color=INK_MUTED, alpha=0.10)
ax.annotate("band BBRsim samples", (np.sqrt(BAND[0] * BAND[1]) / 1e9, 2e-6),
            ha="center", fontsize=8, color=INK_2)
ax.set_xlabel("frequency [GHz]")
ax.set_ylabel("absorptance $D = 1-R$")
ax.set_title("Literature references sit above the band of interest",
             loc="left", color=INK)
ax.legend(fontsize=8, labelcolor=INK_2, loc="upper left")
plt.show()

## 8. What this model does not include

- **The anomalous skin effect.** At 4 K the electron mean free path in
  high-purity copper exceeds the classical skin depth, so the local
  conductivity that Drude assumes stops being the right quantity. The
  correction raises absorptance above the classical plateau, and measured
  cryogenic losses for high-RRR copper do sit above it. BBRsim uses classical
  Drude deliberately, and this is the first thing to revisit once the
  experimental validation of objective O2 produces its own data.
- **Surface preparation.** Oxide layers, machining damage and roughness all
  raise loss, and none of them appear in a bulk model parameterised by purity
  alone. The simulation treats reflection as specular, which the Rayleigh
  criterion justifies at these wavelengths, but only for a clean surface.
- **Anything below 10 GHz or above 20 THz.** The tabulated grid stops there
  because the Planck sampler does.

## Where to go next

- Derivation and constants: `docs/physics/copper_reflectance_model.md`
- Technical brief with figures: `docs/physics/copper_reflectance_brief.tex`
- The model in C++: `BuildDrudeMaterial` in `library/include/BBRMaterials.hh`
- The same model in Python: `tools/python/bbrsim/physics.py`
- Tests of the Python model against analytic limits and reference values: `tools/python/tests/test_physics.py`
- Standalone figure script: `tools/plot_cu_reflectance.py`